In [1]:
!pip install fastapi uvicorn scikit-learn pandas numpy joblib

In [2]:
import numpy as np
import pandas as pd

np.random.seed(42)

n = 1500

age = np.random.randint(18, 71, n)
annual_income = np.random.randint(150000, 2000001, n)
has_existing_policy = np.random.randint(0, 2, n)

# Generate illustrative purchase probabilities
score = (
    -4
    + 0.065 * (age - 35)
    + 0.0000015 * (annual_income - 600000)
    - 0.8 * has_existing_policy
)

probability = 1 / (1 + np.exp(-score))
purchased = np.random.binomial(1, probability)

df = pd.DataFrame({
    "Age": age,
    "AnnualIncome": annual_income,
    "HasExistingPolicy": has_existing_policy,
    "Purchased": purchased
})

df.to_csv("insurance_data.csv", index=False)

print("Dataset created successfully!")
display(df.head())
print("\nDataset shape:", df.shape)
print("\nTarget distribution:")
print(df["Purchased"].value_counts())

Dataset created successfully!


,Age,AnnualIncome,HasExistingPolicy,Purchased
0,56,1455289,1,0
1,69,1640077,1,1
2,46,693351,1,0
3,32,217642,1,0
4,60,240093,1,0



Dataset shape: (1500, 4)

Target distribution:
Purchased
0    1379
1     121
Name: count, dtype: int64


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
import joblib

# Input features and target
X = df[
    ["Age", "AnnualIncome", "HasExistingPolicy"]
]
y = df["Purchased"]

# Split dataset: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Scaling and model training
model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

# Evaluate the model
y_pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Save the complete pipeline
joblib.dump(model, "model.pkl")

print("\nModel saved successfully as model.pkl")

Accuracy: 92.0 %

Classification Report:
              precision    recall  f1-score   support

           0       0.92      1.00      0.96       276
           1       0.00      0.00      0.00        24

    accuracy                           0.92       300
   macro avg       0.46      0.50      0.48       300
weighted avg       0.85      0.92      0.88       300


Confusion Matrix:
[[276   0]
 [ 24   0]]

Model saved successfully as model.pkl


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [4]:
import os

os.makedirs("templates", exist_ok=True)
os.makedirs("static", exist_ok=True)

print("Project folders created successfully!")

Project folders created successfully!


In [5]:
%%writefile main.py
from pathlib import Path

import joblib
import pandas as pd
from fastapi import FastAPI
from fastapi.responses import HTMLResponse
from fastapi.staticfiles import StaticFiles
from fastapi.templating import Jinja2Templates
from fastapi import Request
from pydantic import BaseModel, Field

BASE_DIR = Path(__file__).resolve().parent

model = joblib.load(BASE_DIR / "model.pkl")

app = FastAPI(title="Insurance Purchase Prediction")

app.mount(
    "/static",
    StaticFiles(directory=str(BASE_DIR / "static")),
    name="static"
)

templates = Jinja2Templates(
    directory=str(BASE_DIR / "templates")
)


class CustomerInput(BaseModel):
    age: int = Field(ge=18, le=100)
    annual_income: float = Field(ge=0, le=100000000)
    has_existing_policy: bool


@app.get("/", response_class=HTMLResponse)
def home(request: Request):
    return templates.TemplateResponse(
        request=request,
        name="index.html",
        context={}
    )


@app.get("/health")
def health():
    return {"status": "running"}


@app.post("/predict")
def predict(customer: CustomerInput):
    features = pd.DataFrame([{
        "Age": customer.age,
        "AnnualIncome": customer.annual_income,
        "HasExistingPolicy": int(
            customer.has_existing_policy
        )
    }])

    prediction = int(model.predict(features)[0])
    probabilities = model.predict_proba(features)[0]
    purchase_probability = float(probabilities[1] * 100)

    return {
        "prediction": prediction,
        "result": (
            "Likely to purchase insurance"
            if prediction == 1
            else "Unlikely to purchase insurance"
        ),
        "purchase_probability": round(
            purchase_probability, 2
        )
    }

Writing main.py


In [6]:
%%writefile templates/index.html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Insurance Purchase Prediction</title>
    <link rel="stylesheet" href="/static/style.css">
</head>
<body>
    <main class="container">
        <section class="card">
            <div class="icon">🛡️</div>
            <h1>Insurance Purchase Prediction</h1>
            <p class="subtitle">
                Enter customer details to estimate insurance purchase likelihood.
            </p>

            <form id="predictionForm">
                <label for="age">Customer Age</label>
                <input id="age" type="number" min="18" max="100"
                       placeholder="Enter age (18–100)" required>

                <label for="income">Annual Income (₹)</label>
                <input id="income" type="number" min="0"
                       max="100000000" step="0.01"
                       placeholder="e.g. 600000" required>

                <label for="policy">Already has an insurance policy?</label>
                <select id="policy" required>
                    <option value="false">No</option>
                    <option value="true">Yes</option>
                </select>

                <button type="submit" id="submitButton">
                    Predict Purchase
                </button>
            </form>

            <div id="result" class="result" role="status"
                 aria-live="polite" hidden></div>

            <p class="disclaimer">
                Educational demonstration only. Predictions are not
                financial or insurance advice.
            </p>
        </section>
    </main>

    <script src="/static/script.js"></script>
</body>
</html>

Writing templates/index.html


In [7]:
%%writefile static/style.css
* {
    box-sizing: border-box;
}

body {
    margin: 0;
    min-height: 100vh;
    font-family: Arial, sans-serif;
    background: linear-gradient(135deg, #e0eaff, #f5f3ff);
    color: #1f2937;
    display: flex;
    justify-content: center;
    align-items: center;
    padding: 24px;
}

.container {
    width: 100%;
    max-width: 520px;
}

.card {
    background: white;
    padding: 32px;
    border-radius: 18px;
    box-shadow: 0 12px 35px rgba(30, 50, 90, 0.12);
}

.icon {
    font-size: 40px;
    text-align: center;
}

h1 {
    text-align: center;
    font-size: 27px;
    margin-bottom: 10px;
    color: #243b70;
}

.subtitle {
    text-align: center;
    color: #64748b;
    line-height: 1.5;
    margin-bottom: 28px;
}

label {
    display: block;
    margin: 17px 0 8px;
    font-weight: 600;
}

input, select {
    width: 100%;
    padding: 12px;
    border: 1px solid #cbd5e1;
    border-radius: 8px;
    font-size: 15px;
    background: white;
}

input:focus, select:focus {
    outline: 2px solid #7898f5;
    border-color: #496de0;
}

button {
    width: 100%;
    padding: 14px;
    margin-top: 24px;
    border: none;
    border-radius: 9px;
    background: #3459c7;
    color: white;
    font-size: 16px;
    font-weight: bold;
    cursor: pointer;
}

button:hover {
    background: #2546a8;
}

button:disabled {
    opacity: 0.65;
    cursor: wait;
}

.result {
    margin-top: 22px;
    padding: 16px;
    border-radius: 9px;
    line-height: 1.7;
    overflow-wrap: anywhere;
}

.result.success {
    background: #e8f7ee;
    color: #166534;
}

.result.warning {
    background: #fff4e5;
    color: #92400e;
}

.result.error {
    background: #feecec;
    color: #991b1b;
}

.disclaimer {
    margin-top: 24px;
    font-size: 12px;
    line-height: 1.5;
    text-align: center;
    color: #64748b;
}

Writing static/style.css


In [8]:
%%writefile static/script.js
const form = document.getElementById("predictionForm");
const resultBox = document.getElementById("result");
const submitButton = document.getElementById("submitButton");

function showMessage(message, type) {
    resultBox.hidden = false;
    resultBox.className = `result ${type}`;
    resultBox.textContent = message;
}

form.addEventListener("submit", async (event) => {
    event.preventDefault();

    const age = Number(document.getElementById("age").value);
    const income = Number(document.getElementById("income").value);
    const policy = document.getElementById("policy").value;

    if (!Number.isInteger(age) || age < 18 || age > 100) {
        showMessage("Please enter an age from 18 to 100.", "error");
        return;
    }

    if (!Number.isFinite(income) || income < 0 ||
        income > 100000000) {
        showMessage("Please enter a valid annual income.", "error");
        return;
    }

    submitButton.disabled = true;
    submitButton.textContent = "Predicting...";

    try {
        const response = await fetch("/predict", {
            method: "POST",
            headers: {
                "Content-Type": "application/json"
            },
            body: JSON.stringify({
                age: age,
                annual_income: income,
                has_existing_policy: policy === "true"
            })
        });

        const data = await response.json();

        if (!response.ok) {
            throw new Error(
                data.detail
                    ? JSON.stringify(data.detail)
                    : "Unable to generate prediction."
            );
        }

        const type = data.prediction === 1 ? "success" : "warning";

        showMessage(
            `Result: ${data.result}\n` +
            `Estimated purchase probability: ` +
            `${data.purchase_probability}%`,
            type
        );
    } catch (error) {
        showMessage(`Error: ${error.message}`, "error");
    } finally {
        submitButton.disabled = false;
        submitButton.textContent = "Predict Purchase";
    }
});

Writing static/script.js


In [9]:
import subprocess
import time
import requests
from google.colab.output import eval_js

# Start the FastAPI server
server = subprocess.Popen([
    "python", "-m", "uvicorn",
    "main:app",
    "--host", "0.0.0.0",
    "--port", "8000"
])

# Wait for the server to start
for attempt in range(30):
    try:
        response = requests.get(
            "http://127.0.0.1:8000/health",
            timeout=2
        )
        if response.status_code == 200:
            print("FastAPI server started successfully!")
            print(response.json())
            break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError(
        "Server did not start. Check the preceding error messages."
    )

# Open the application through Colab's port proxy
app_url = eval_js("google.colab.kernel.proxyPort(8000)")
print("\nOpen your application here:")
print(app_url)

FastAPI server started successfully!
{'status': 'running'}

Open your application here:
https://8000-m-s-kkb-usw1b0-38cusk8858rnu-b.us-west1-0.prod.colab.dev
